[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/potential_capacitance.ipynb)

# Electrostatic potential and capacitance full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the level of the course's worked problems) and **tier 3** stretch problems (run a formula
backwards, combine two ideas, spot the trap). Each tier is harder than the last.

This notebook also has the course's **simulations and sums**: a point charge's potential built step by step from far
away, equipotentials traced a small step at a time, an electron crossing two gaps at the same voltage, capacitors in
series and in parallel, a capacitor charged a spoonful at a time, and the camera flash. You can move five of them
yourself: the **second charge** of the equipotential picture, the **gap** the electron crosses, the **two
capacitors**, the **number of spoonfuls**, and the **flash's capacitor and voltage**.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Give the answer
in the unit the question asks for, to two decimals or more. Big and small numbers in E notation: `8.99e9` for
8.99 × 10⁹, `1.6e-19` for 1.6 × 10⁻¹⁹ (`8.99*10^9` works too). You can type the arithmetic: `8.99e9*2e-9/0.15`,
`0.5*10e-6*100**2`, `sqrt(2*1.6e-19*500/9.11e-31)/1e7`. Numbers without thousands separators: `1798`, not `1,798`. For
several answers, separate them with commas, in the order the question lists them (the box's label shows the order). You
can also put your answer between the quotes in a cell's last line, `practice("P1.4", "")`, and run the cell: it checks
straight away.

**The cheat sheet.** Every formula and constant from the course on two pages:
[electrostatic potential and capacitance cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/potential_capacitance_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the constants, the simulator and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, IntSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # charges and lines in chalk, labels dim, ghost lines
SKY, MINT, CORAL, VIOLET, GOLD = "#6fb6bb", "#a0c27c", "#e3896b", "#b49ad6", "#e8c56b"
# velocity SKY · gravity MINT · force and field CORAL · energy VIOLET · charges and lines CHALK · the one mark GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="simulated · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- the constants (the course's rounded values; the same in every question) ----
K = 8.99e9               # N m²/C², Coulomb's constant k
EPS0 = 8.85e-12          # C²/(N m²), epsilon nought (1/(4π ε₀) = 8.99 × 10⁹)
E = 1.60e-19             # C, the size of one electron's charge
ME = 9.11e-31            # kg, an electron's mass
g = 9.8                  # m/s²
E_SPARK = 3.0e6          # V/m: dry air at sea level sparks at about this field

# ---- the course's simulator: fourth-order Runge–Kutta, a fixed step at a time ----
def simulate(deriv, state, t_end, dt=1e-3, stop=None):
    """deriv(t, s) -> ds/dt as a tuple; state a tuple of numbers. Returns the list of (t, state) samples from 0 to
    t_end, or up to the first step where stop(t, s) is True."""
    t, s = 0.0, tuple(float(v) for v in state)
    out = [(t, s)]
    for _ in range(int(round(t_end / dt))):
        k1 = deriv(t, s)
        k2 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k1)))
        k3 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k2)))
        k4 = deriv(t + dt, tuple(a + dt * b for a, b in zip(s, k3)))
        s = tuple(a + dt / 6 * (b1 + 2 * b2 + 2 * b3 + b4) for a, b1, b2, b3, b4 in zip(s, k1, k2, k3, k4))
        t = t + dt
        out.append((t, s))
        if stop is not None and stop(t, s): return out
    return out

def close(a, b, rel=1e-4):
    """True when a simulated or summed value matches the formula (to a ten-thousandth by default)."""
    return abs(a - b) <= rel * max(abs(a), abs(b))

# ---- the course's formulas ----
def potential(Q, r): return K * Q / r                                    # V = kQ/r
def field_point(Q, r): return K * Q / r ** 2                             # E = kQ/r²
def pe_pair(q1, q2, r): return K * q1 * q2 / r                           # U = k q₁q₂/r
def dipole_v(p, r, theta_deg): return K * p * math.cos(math.radians(theta_deg)) / r ** 2   # far away
def pe_dipole(p, Ef, theta_deg): return -p * Ef * math.cos(math.radians(theta_deg))   # U = −pE cos θ
def cap_plates(A, d, kappa=1.0): return kappa * EPS0 * A / d             # C = κ ε₀ A/d
def series(*cs): return 1 / sum(1 / c for c in cs)                       # 1/C = 1/C₁ + 1/C₂ + …
def parallel(*cs): return sum(cs)                                        # C = C₁ + C₂ + …
def energy(C, V): return 0.5 * C * V * V                                 # U = ½ C V²
def density(Ef, kappa=1.0): return 0.5 * kappa * EPS0 * Ef * Ef          # u = ½ ε₀ E²

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "cos": math.cos, "sin": math.sin, "acos": math.acos}

def calc(text):
    """Plain arithmetic only: numbers (8.99e9 too), + − × ÷, powers, brackets, sqrt, cos, sin, acos (radians) and pi."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id in ("pi", "π"): return math.pi
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = (r"\s*(N m²/C|N/C|V/m|J/m³|J/m3|N m|C m|µF|uF|pF|nF|µC|uC|nC|mC|C|F|J|nJ|µJ|uJ|V|W|N|m/s|cm|mm|m²|m2|m|s|%|°|"
        r"degrees?|times)\s*$")

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    s = s.replace("√", "sqrt")
    t = re.sub(UNIT, "", s).strip()
    try: return calc(t)
    except Exception: return s

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the course's questions and answers) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Electric potential',
     "q": 'Moving a +4 µC charge from A to B takes 2.0 × 10⁻⁴ J of work. What is the potential difference between B and A, in volts?',
     "parts": ['potential difference (V)'],
     "answer": [50.00000000000001], "tol": [0.005], "function": '/',
     "hint": 'potential is energy per coulomb: W ÷ q',
     "working": 'V = W/q = 2.0 × 10⁻⁴ / 4 × 10⁻⁶ = 50 V'},
    {"id": 'P1.2', "tier": 1, "group": 'Electric potential',
     "q": 'What is the potential 15 cm from a +2 nC point charge, in volts?',
     "parts": ['potential (V)'],
     "answer": [119.86666666666667], "tol": [0.005], "function": '—',
     "hint": 'V = k Q ÷ r (not r²)',
     "working": 'V = kQ/r = 8.99 × 10⁹ × 2 × 10⁻⁹ / 0.15 = 119.8667 V'},
    {"id": 'P1.3', "tier": 1, "group": 'Potential of many charges',
     "q": 'Charges of +4 nC and +2 nC sit 30 cm apart. What is the potential at the midpoint, in volts?',
     "parts": ['potential at the midpoint (V)'],
     "answer": [359.6], "tol": [0.005], "function": '—',
     "hint": 'add the two potentials; no arrows needed',
     "working": 'V = k(4 + 2) × 10⁻⁹ / 0.15 = 359.6000 V: potentials add as plain numbers'},
    {"id": 'P1.4', "tier": 1, "group": 'Potential of many charges',
     "q": 'Two plates 6 mm apart have 24 V between them. What is the field between them, in volts per metre?',
     "parts": ['field (V/m)'],
     "answer": [4000.0], "tol": [0.005], "function": '/',
     "hint": 'the field is how fast the potential changes: V ÷ d',
     "working": 'E = V/d = 24 / 0.006 = 4000 V/m'},
    {"id": 'P1.5', "tier": 1, "group": 'Potential energy',
     "q": 'Charges of +2 µC and −5 µC sit 50 cm apart. What is their potential energy, in joules?',
     "parts": ['potential energy (J)'],
     "answer": [-0.17980000000000002], "tol": [0.005], "function": '—',
     "hint": 'U = k q1 q2 ÷ r, with the signs',
     "working": 'U = k q₁q₂/r = 8.99 × 10⁹ × (2 × 10⁻⁶)(−5 × 10⁻⁶) / 0.5 = -0.1798 J (negative: they are bound)'},
    {"id": 'P1.6', "tier": 1, "group": 'Potential energy',
     "q": 'An electron (charge 1.60 × 10⁻¹⁹ C) is pulled from rest through a potential difference of 200 V. How much energy does it gain, in units of 10⁻¹⁷ J?',
     "parts": ['energy (× 10⁻¹⁷ J)'],
     "answer": [3.1999999999999993], "tol": [0.005], "function": '—',
     "hint": 'energy = charge × potential difference',
     "working": 'U = qV = 1.60 × 10⁻¹⁹ × 200 = 3.2 × 10⁻¹⁷ J'},
    {"id": 'P1.7', "tier": 1, "group": 'Conductors',
     "q": 'A charged metal surface carries 1.77 µC per square metre. What is the field just outside it, in units of 10⁵ N/C?',
     "parts": ['field (× 10⁵ N/C)'],
     "answer": [2.0], "tol": [0.005], "function": '—',
     "hint": 'E = σ ÷ ε₀ just outside',
     "working": 'E = σ/ε₀ = 1.77 × 10⁻⁶ / 8.85 × 10⁻¹² = 200000.0 N/C'},
    {"id": 'P1.8', "tier": 1, "group": 'Capacitors',
     "q": 'A 47 µF capacitor is connected to 12 V. How much charge does it hold, in µC?',
     "parts": ['charge (µC)'],
     "answer": [563.9999999999999], "tol": [0.005], "function": '—',
     "hint": 'capacitance is charge per volt: Q = C V',
     "working": 'Q = CV = 47 × 10⁻⁶ × 12 = 564 × 10⁻⁶ C = 564 µC'},
    {"id": 'P1.9', "tier": 1, "group": 'Capacitors',
     "q": 'Two metal plates of area 0.05 m² face each other 2 mm apart, with air between. What is the capacitance, in pF?',
     "parts": ['capacitance (pF)'],
     "answer": [221.25], "tol": [0.005], "function": '—',
     "hint": 'C = ε₀ A ÷ d; 1 pF = 1e-12 F',
     "working": 'C = ε₀A/d = 8.85 × 10⁻¹² × 0.05 / 0.002 = 2.2125e-10 F = 221.2500 pF'},
    {"id": 'P1.10', "tier": 1, "group": 'Series and parallel',
     "q": 'Capacitors of 4 µF and 6 µF are joined in parallel. What is their combined capacitance, in µF?',
     "parts": ['combined capacitance (µF)'],
     "answer": [10.0], "tol": [0.005], "function": '—',
     "hint": 'same voltage across each: the charges add',
     "working": 'in parallel, the capacitances add: 4 + 6 = 10 µF'},
    {"id": 'P2.1', "tier": 2, "group": 'Electric potential',
     "q": "The potential 20 cm from a small charged ball is 450 V. What is the ball's charge, in nC?",
     "parts": ['charge (nC)'],
     "answer": [10.011123470522802], "tol": [0.005], "function": '—',
     "hint": 'run V = kQ/r backwards: Q = V r ÷ k',
     "working": 'Q = V r / k = 450 × 0.2 / 8.99 × 10⁹ = 1.0011e-08 C = 10.0111 nC'},
    {"id": 'P2.2', "tier": 2, "group": 'Potential of many charges',
     "q": 'A dipole has p = 4 × 10⁻¹² C m. Using the far-away formula, what is the potential 10 cm from it, in a direction 60° from its axis, in volts?',
     "parts": ['potential (V)'],
     "answer": [1.798], "tol": [0.005], "function": 'cos',
     "hint": 'V = k p cos θ ÷ r²; cos 60° = 0.5',
     "working": 'V = kp cos θ / r² = 8.99 × 10⁹ × 4 × 10⁻¹² × 0.5 / 0.01 = 1.7980 V'},
    {"id": 'P2.3', "tier": 2, "group": 'Potential of many charges',
     "q": '+6 nC sits at 0 and −2 nC at 0.4 m. Where on the line is the potential zero? Give the point between them and the point beyond the −2 nC, in cm from the +6 nC.',
     "parts": ['between (cm)', 'beyond (cm)'],
     "answer": [30.0, 60.0], "tol": [0.005, 0.005], "function": '—',
     "hint": 'set k·6/x equal to k·2/(distance to the −2 nC)',
     "working": 'between: 6/x = 2/(0.4 − x) → x = 0.3 m; beyond: 6/x = 2/(x − 0.4) → x = 0.6 m'},
    {"id": 'P2.4', "tier": 2, "group": 'Potential energy',
     "q": 'Charges of +1 µC, +1 µC and −2 µC sit at the corners of a triangle with 20 cm sides. What is the potential energy of the group, in joules?',
     "parts": ['potential energy (J)'],
     "answer": [-0.13485], "tol": [0.005], "function": '—',
     "hint": 'add k q q ÷ r over every pair, once each',
     "working": 'three pairs: k(1·1 + 1·(−2) + 1·(−2)) × 10⁻¹² / 0.2 = -0.1348 J'},
    {"id": 'P2.5', "tier": 2, "group": 'Conductors',
     "q": 'Metal balls of radius 12 cm and 3 cm are joined by a long thin wire and share 30 nC. What charge does each carry, in nC, and what is their common potential, in volts?',
     "parts": ['big ball (nC)', 'small ball (nC)', 'potential (V)'],
     "answer": [24.0, 6.0, 1797.9999999999998], "tol": [0.005, 0.005, 0.005], "function": '—',
     "hint": 'joined conductors end at one potential',
     "working": 'the same potential: Q₁/R₁ = Q₂/R₂, so charge goes as the radius: 24 nC and 6 nC; V = k × 24 × 10⁻⁹ / 0.12 = 1798.0000 V'},
    {"id": 'P2.6', "tier": 2, "group": 'Capacitors',
     "q": "When a capacitor's voltage rises from 5 V to 11 V, its charge rises by 30 µC. What is its capacitance, in µF?",
     "parts": ['capacitance (µF)'],
     "answer": [5.0], "tol": [0.005], "function": '—',
     "hint": 'capacitance is charge per volt',
     "working": 'C = ΔQ/ΔV = 30 µC / 6 V = 5 µF'},
    {"id": 'P2.7', "tier": 2, "group": 'Dielectrics',
     "q": 'Plates of area 0.02 m², 0.5 mm apart, are filled with a plastic of dielectric constant 4. What is the capacitance, in pF?',
     "parts": ['capacitance (pF)'],
     "answer": [1416.0], "tol": [0.005], "function": '—',
     "hint": 'the dielectric multiplies ε₀A/d by κ',
     "working": 'C = κ ε₀ A/d = 4 × 8.85 × 10⁻¹² × 0.02 / 0.0005 = 1416.0000 pF'},
    {"id": 'P2.8', "tier": 2, "group": 'Series and parallel',
     "q": 'Capacitors of 2 µF and 3 µF are joined in series across 10 V. What is the combined capacitance (µF), the charge on each (µC), and the voltage across the 2 µF (V)?',
     "parts": ['combined (µF)', 'charge on each (µC)', 'voltage across 2 µF (V)'],
     "answer": [1.2, 12.0, 6.0], "tol": [0.005, 0.005, 0.005], "function": '—',
     "hint": 'in series: the same charge on each',
     "working": '1/C = 1/2 + 1/3 → C = 1.2 µF; Q = 1.2 × 10 = 12 µC on each; V₁ = 12/2 = 6 V'},
    {"id": 'P2.9', "tier": 2, "group": 'Energy stored',
     "q": 'A 10 µF capacitor is charged to 100 V. How much energy does it store, in joules?',
     "parts": ['energy (J)'],
     "answer": [0.05], "tol": [0.005], "function": '—',
     "hint": 'U = ½ C V²',
     "working": 'U = ½CV² = 0.5 × 10 × 10⁻⁶ × 100² = 0.05 J'},
    {"id": 'P2.10', "tier": 2, "group": 'Energy stored',
     "q": 'The field in a region of air is 1.0 × 10⁶ V/m. How much energy is stored in each cubic metre, in joules?',
     "parts": ['energy per cubic metre (J)'],
     "answer": [4.425], "tol": [0.005], "function": '—',
     "hint": 'u = ½ ε₀ E²',
     "working": 'u = ½ε₀E² = 0.5 × 8.85 × 10⁻¹² × (10⁶)² = 4.4250 J/m³'},
    {"id": 'P3.1', "tier": 3, "group": 'Electric potential',
     "q": 'A +2 µC charge is moved from 30 cm to 10 cm from a fixed +5 µC charge, along a curving path. How much work does it take, in joules? (Does the path matter?)',
     "parts": ['work (J)'],
     "answer": [0.5993333333333333], "tol": [0.005], "function": '—',
     "hint": 'only the start and the end count',
     "working": "W = q(V_end − V_start) = k × 10 × 10⁻¹² × (1/0.1 − 1/0.3) = 0.5993 J; the path doesn't matter"},
    {"id": 'P3.2', "tier": 3, "group": 'Potential of many charges',
     "q": 'Near a small charged ball, the potential is 899 V at 10 cm and 817.27 V at 11 cm. Estimate the field between those points from the slope, in volts per metre.',
     "parts": ['field (V/m)'],
     "answer": [8172.727272727263], "tol": [0.005], "function": '—',
     "hint": "the field is the potential's drop per metre",
     "working": 'E ≈ −ΔV/Δr = (899 − 817.27) / 0.01 = 8172.7273 V/m (exactly kQ/r² at √(0.10 × 0.11) m)'},
    {"id": 'P3.3', "tier": 3, "group": 'Potential energy',
     "q": 'An electron (charge 1.60 × 10⁻¹⁹ C, mass 9.11 × 10⁻³¹ kg) starts from rest and crosses a potential difference of 500 V. What is its speed, in units of 10⁷ m/s?',
     "parts": ['speed (× 10⁷ m/s)'],
     "answer": [1.3252591238451432], "tol": [0.005], "function": 'sqrt',
     "hint": 'energy gained = e V; then ½ m v²',
     "working": '½mv² = eV → v = √(2eV/m) = 1.3253e+07 m/s'},
    {"id": 'P3.4', "tier": 3, "group": 'Capacitors',
     "q": 'You want a 1 nF capacitor from two plates 1 mm apart with air between. How big must each plate be, in square metres?',
     "parts": ['plate area (m²)'],
     "answer": [0.11299435028248589], "tol": [0.005], "function": '—',
     "hint": 'run C = ε₀A/d backwards',
     "working": 'A = C d / ε₀ = 10⁻⁹ × 10⁻³ / 8.85 × 10⁻¹² = 0.1130 m²'},
    {"id": 'P3.5', "tier": 3, "group": 'Dielectrics',
     "q": 'A 100 pF capacitor is charged to 50 V, then taken off the battery. A slab with dielectric constant 5 is slid in to fill the gap. What is the new voltage (V), and the new stored energy (nJ)?',
     "parts": ['new voltage (V)', 'new energy (nJ)'],
     "answer": [10.0, 25.0], "tol": [0.005, 0.005], "function": '—',
     "hint": 'which stays the same: Q or V?',
     "working": 'off the battery the charge stays 5 nC; C becomes 500 pF; V = 5 nC / 500 pF = 10 V; U = ½ × 500 pF × 10² = 25 nJ (it was 125 nJ: the slab is pulled in)'},
    {"id": 'P3.6', "tier": 3, "group": 'Dielectrics',
     "q": 'The same 100 pF capacitor stays on its 50 V battery while the slab (dielectric constant 5) is slid in. What is the new charge (nC), and the new stored energy (nJ)?',
     "parts": ['new charge (nC)', 'new energy (nJ)'],
     "answer": [25.000000000000004, 625.0], "tol": [0.005, 0.005], "function": '—',
     "hint": 'the battery holds the voltage fixed',
     "working": 'on the battery V stays 50 V; C becomes 500 pF; Q = 500 pF × 50 V = 25 nC; U = ½ × 500 pF × 50² = 625 nJ'},
    {"id": 'P3.7', "tier": 3, "group": 'Series and parallel',
     "q": 'Capacitors of 3 µF and 6 µF are in series, and that pair is in parallel with a 4 µF capacitor, across 12 V. What is the combined capacitance (µF), and the charge on the 3 µF capacitor (µC)?',
     "parts": ['combined (µF)', 'charge on the 3 µF (µC)'],
     "answer": [6.0, 24.0], "tol": [0.005, 0.005], "function": '—',
     "hint": 'reduce the pair first',
     "working": '3 and 6 in series: 2 µF; with 4 in parallel: 6 µF; the pair has 12 V across it: 2 µF × 12 V = 24 µC on each of the 3 µF and the 6 µF'},
    {"id": 'P3.8', "tier": 3, "group": 'Energy stored',
     "q": 'A flash capacitor of 120 µF is charged to 330 V. How much energy does it store (J), and how long would a charger supplying 2 W take to fill it (s)?',
     "parts": ['energy (J)', 'charging time (s)'],
     "answer": [6.534000000000001, 3.2670000000000003], "tol": [0.005, 0.005], "function": '—',
     "hint": 'energy first, then time = energy ÷ power',
     "working": 'U = ½ × 120 × 10⁻⁶ × 330² = 6.5340 J; t = U/P = 3.2670 s'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g_, w, t) for g_, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else "✓ all right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
            print(textwrap.fill(f"Hint: {q['hint']}", 110))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g_, w, t) for g_, w, t in zip(got, answers(k), Q[k]["tol"]))

assert round(1 / (4 * math.pi * EPS0) / 1e9, 2) == 8.99
print("Ready: k =", K, "N m²/C²; epsilon nought =", EPS0, "C²/(N m²)")

## Tier 1 · Warm-up

One idea, one step: potential from work, the potential of a point charge, two potentials added, the field from a
voltage, the energy of two charges, an electron's energy, the field outside a conductor, a capacitor's charge, a
parallel-plate capacitor, and two capacitors in parallel.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each formula from the course is one short function in the first cell: `potential(Q, r)`, `pe_pair(q1, q2, r)`,
`cap_plates(A, d)`, `series(C1, C2)`, `energy(C, V)` and the rest. Work in coulombs, metres, volts and farads (nC is
`1e-9`, µF is `1e-6`, pF is `1e-12`, cm is `0.01`), then change the unit at the end. Each line below computes one
warm-up answer and checks it against the practice set.

In [ ]:
print(f"P1.2  V = {potential(2e-9, 0.15):.2f} V;  P1.9  C = {cap_plates(0.05, 2e-3) * 1e12:.2f} pF")
assert agrees("P1.1", [2e-4 / 4e-6]) and agrees("P1.2", [potential(2e-9, 0.15)])
assert agrees("P1.3", [potential(4e-9, 0.15) + potential(2e-9, 0.15)]) and agrees("P1.4", [24 / 6e-3])
assert agrees("P1.5", [pe_pair(2e-6, -5e-6, 0.5)]) and agrees("P1.6", [E * 200 / 1e-17])
assert agrees("P1.7", [1.77e-6 / EPS0 / 1e5]) and agrees("P1.8", [47e-6 * 12 * 1e6])
assert agrees("P1.9", [cap_plates(0.05, 2e-3) * 1e12]) and agrees("P1.10", [parallel(4, 6)])
print("Every tier 1 answer, worked out in Python, matches the practice set.")

## A point charge's potential, built step by step

Bring a test charge in from far away towards a charge Q, in small steps. On each step the push on one coulomb barely
changes; its work is the push times the step. Add up all the steps and you get k Q over r. Slide the number of steps:
even a few hundred steps land on the formula.

In [ ]:
def built_potential(Q=1e-9, r=0.1, far=1e6, n=2000):
    rs = np.geomspace(r, far, n + 1)
    mid = (rs[:-1] + rs[1:]) / 2
    pieces = K * Q / mid ** 2 * np.diff(rs)               # the push on one coulomb at the step's middle, times the step
    return float(np.sum(pieces)), rs, np.cumsum(pieces[::-1])[::-1]

def potential_picture(steps=2000):
    V_sum, rs, running = built_potential(n=int(steps))
    fig, ax = plt.subplots(figsize=(10, 4.2))
    fig.subplots_adjust(left=0.09, right=0.98, top=0.84, bottom=0.17)
    x = np.linspace(0.1, 1.0, 200)
    ax.plot(x, potential(1e-9, x), color=DIM, label="k Q / r")
    ax.plot(rs[:-1], running, color=GOLD, lw=1.6, label="added up, step by step")
    ax.set_xlim(0.08, 1.0); ax.set_xlabel("distance from the charge (m)"); ax.set_ylabel("V (volts)")
    ax.legend(frameon=False)
    fig.suptitle(f"{int(steps)} steps: {V_sum:.4f} V at 10 cm; k Q / r = {potential(1e-9, 0.1):.4f} V", x=0.09, ha="left",
                 color=GOLD)
    footnote(fig, "summed step by step from 10⁶ m"); plt.show()
    return V_sum

assert close(potential_picture(2000), potential(1e-9, 0.1), 1e-4)

In [ ]:
interact(potential_picture, steps=IntSlider(value=2000, min=50, max=20000, step=50, description="steps"));

## Equipotentials, traced

An equipotential is traced the way the course traces it: start on a point, take a small step at right angles to the
field (so the potential doesn't change), look again, step again, until the loop closes. Here a +1 charge sits on the
left and a second charge on the right (units where k = 1). The potential is read back along each loop: it stays the
same. Field lines would cross every loop square on. Slide the second charge from −2 to +2.

In [ ]:
def vpic(charges, x, y): return sum(q / math.hypot(x - cx, y - cy) for q, cx, cy in charges)

def efield(charges, x, y):
    ex = ey = 0.0
    for q, cx, cy in charges:
        dx, dy = x - cx, y - cy
        d3 = (dx * dx + dy * dy) ** 1.5
        ex += q * dx / d3; ey += q * dy / d3
    return ex, ey

def trace_equipotential(charges, x0, ds=0.004, length=40.0):
    """Step at right angles to the field (the field turned a quarter turn) until the loop closes."""
    def f(t, s):
        ex, ey = efield(charges, s[0], s[1]); m = math.hypot(ex, ey)
        return (-ey / m, ex / m)
    def stop(t, s): return t > 0.08 and math.hypot(s[0] - x0, s[1]) < 0.02
    return [p[1] for p in simulate(f, (x0, 0.0), length, ds, stop)]

def start_on_axis(charges, level):
    lo, hi = -0.95, -0.05                                 # between the +1 at x = −1 and the middle
    for _ in range(80):
        mid = (lo + hi) / 2
        if (vpic(charges, mid, 0) - level) > 0: lo = mid
        else: hi = mid
    return (lo + hi) / 2

def equipotential_picture(second=-1.0, levels=(2.0, 1.0, 0.5)):
    charges = [(1.0, -1.0, 0.0), (second, 1.0, 0.0)]
    fig, ax = plt.subplots(figsize=(9.5, 6.0))
    fig.subplots_adjust(left=0.04, right=0.98, top=0.88, bottom=0.04)
    drift = 0.0
    for lv in levels:
        x0 = start_on_axis(charges, lv)
        pts = trace_equipotential(charges, x0)
        xs, ys = [p[0] for p in pts], [p[1] for p in pts]
        ax.plot(xs, ys, color=GOLD if lv == levels[0] else CHALK, lw=1.6)
        drift = max(drift, max(abs(vpic(charges, x, y) - lv) / lv for x, y in pts))
    for q, cx, cy in charges:
        if q == 0: continue
        ax.plot([cx], [cy], "o", ms=10 + 6 * abs(q), color=CORAL if q > 0 else SKY)
        ax.text(cx, cy + 0.3, f"{q:+.2g}", color=CHALK, ha="center")
    ax.set_xlim(-3.2, 3.2); ax.set_ylim(-2.2, 2.2); ax.set_aspect("equal"); ax.grid(False)
    ax.set_xticks([]); ax.set_yticks([]); [s_.set_visible(False) for s_ in ax.spines.values()]
    fig.suptitle(f"potential read back along every loop: the same to {drift:.0e}", x=0.04, ha="left", color=GOLD)
    footnote(fig, "traced step by step, at right angles to the field"); plt.show()
    return drift

assert equipotential_picture(-1.0) < 1e-4

In [ ]:
interact(equipotential_picture, second=FloatSlider(value=-1.0, min=-2.0, max=2.0, step=0.25, description="second q"));

## Tier 2 · Core

Several steps each: a charge from its potential, a dipole's potential, where the potential is zero, three charges'
energy, two joined metal balls, a capacitor from a change in charge, a filled capacitor, capacitors in series, the
energy stored, and the energy in a cubic metre of field.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

Potentials and energies are plain numbers: add them with their signs, no arrows. For a group of charges, add k q q / r
over every pair once. In series, the same charge sits on each capacitor, so each voltage is Q / C.

In [ ]:
assert agrees("P2.1", [450 * 0.2 / K * 1e9]) and agrees("P2.2", [dipole_v(4e-12, 0.1, 60)])
assert agrees("P2.3", [0.4 * 6 / 8 * 100, 0.4 * 6 / 4 * 100])
pairs = [(1e-6, 1e-6), (1e-6, -2e-6), (1e-6, -2e-6)]
print(f"P2.4  U = {sum(pe_pair(a, b, 0.2) for a, b in pairs):.4f} J")
assert agrees("P2.4", [sum(pe_pair(a, b, 0.2) for a, b in pairs)])
assert agrees("P2.5", [30 * 0.12 / 0.15, 30 * 0.03 / 0.15, potential(24e-9, 0.12)]) and agrees("P2.6", [30 / 6])
assert agrees("P2.7", [cap_plates(0.02, 5e-4, 4) * 1e12])
C_s = series(2e-6, 3e-6)
assert agrees("P2.8", [C_s * 1e6, C_s * 10 * 1e6, C_s * 10 / 2e-6]) and agrees("P2.9", [energy(10e-6, 100)])
assert agrees("P2.10", [density(1e6)])
print("Every tier 2 answer, worked out in Python, matches the practice set.")

## An electron across two gaps at the same voltage

An electron starts from rest at one plate and crosses to the other, step by step. Make the gap wider at the same 300 V:
the field is weaker, but it pushes for longer. The electron takes longer and arrives just as fast: √(2eV/m).

In [ ]:
def cross(gap, V=300.0):
    a = E * (V / gap) / ME
    run = simulate(lambda t, s: (s[1], a), (0.0, 0.0), 1e-7, 1e-12, lambda t, s: s[0] >= gap)
    return run[-1][0], run[-1][1][1]

def gap_picture(gap_cm=1.0, V=300.0):
    t1, v1 = cross(0.01, V); t2, v2 = cross(gap_cm / 100, V)
    fig, ax = plt.subplots(figsize=(9.5, 3.6))
    fig.subplots_adjust(left=0.25, right=0.97, top=0.78, bottom=0.2)
    ax.barh([1, 0], [v1 / 1e6, v2 / 1e6], color=[DIM, GOLD]); ax.set_yticks([1, 0])
    ax.set_yticklabels([f"1 cm gap: {t1 * 1e9:.2f} ns", f"{gap_cm:g} cm gap: {t2 * 1e9:.2f} ns"])
    ax.set_xlabel("speed on arrival (million m/s)")
    fig.suptitle(f"√(2eV/m) = {math.sqrt(2 * E * V / ME) / 1e6:.3f} million m/s, whatever the gap", x=0.03, ha="left",
                 color=GOLD)
    footnote(fig); plt.show()
    return v1, v2

v_a, v_b = gap_picture(2.0)
assert close(v_a, v_b, 1e-3) and close(v_a, math.sqrt(2 * E * 300 / ME), 1e-3)

In [ ]:
interact(gap_picture, gap_cm=FloatSlider(value=2.0, min=0.5, max=5.0, step=0.5, description="gap (cm)"),
         V=FloatSlider(value=300.0, min=50.0, max=1000.0, step=50.0, description="V (volts)"));

## Tier 3 · Stretch

Run a formula backwards, combine two ideas, spot the trap: work along a curving path, the field from a potential's
slope, an electron's speed, a plate's area for 1 nF, a slab slid in with the battery off and then on, a mixed network,
and a flash capacitor.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

The path never enters the work: only the potentials at the start and the end. Off the battery, the charge stays; on
it, the voltage stays. Reduce a network one pair at a time.

In [ ]:
assert agrees("P3.1", [2e-6 * (potential(5e-6, 0.1) - potential(5e-6, 0.3))])
assert agrees("P3.2", [(potential(10e-9, 0.10) - potential(10e-9, 0.11)) / 0.01])
assert agrees("P3.3", [math.sqrt(2 * E * 500 / ME) / 1e7]) and agrees("P3.4", [1e-9 * 1e-3 / EPS0])
Q0 = 100e-12 * 50
assert agrees("P3.5", [Q0 / 500e-12, energy(500e-12, Q0 / 500e-12) * 1e9])
assert agrees("P3.6", [500e-12 * 50 * 1e9, energy(500e-12, 50) * 1e9])
C_net = parallel(series(3e-6, 6e-6), 4e-6)
assert agrees("P3.7", [C_net * 1e6, series(3e-6, 6e-6) * 12 * 1e6])
assert agrees("P3.8", [energy(120e-6, 330), energy(120e-6, 330) / 2])
print("Every tier 3 answer, worked out in Python, matches the practice set.")

## Capacitors in series and in parallel

Two capacitors on a 10 V battery. In series, the same charge sits on each, and the voltages add to 10 V. In parallel,
each has the full 10 V, and the charges add. Slide the two capacitors and watch the totals: series is always smaller
than the smaller one, parallel always bigger than the bigger one.

In [ ]:
def combine_picture(C1_uF=2.0, C2_uF=3.0, V=10.0):
    C1, C2 = C1_uF * 1e-6, C2_uF * 1e-6
    Cs, Cp = series(C1, C2), parallel(C1, C2)
    Q = Cs * V
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.78, bottom=0.18, wspace=0.35)
    a1.bar(["C₁", "C₂"], [Q / C1, Q / C2], color=[CHALK, DIM]); a1.set_ylabel("volts"); a1.set_title(
        f"series: {Q * 1e6:.2f} µC on each", color=CHALK)
    a2.bar(["C₁", "C₂"], [C1 * V * 1e6, C2 * V * 1e6], color=[CHALK, DIM]); a2.set_ylabel("µC"); a2.set_title(
        f"parallel: {V:g} V across each", color=CHALK)
    fig.suptitle(f"series {Cs * 1e6:.3f} µF · parallel {Cp * 1e6:.3f} µF", x=0.08, ha="left", color=GOLD)
    footnote(fig, "computed"); plt.show()
    return Cs, Cp, Q / C1 + Q / C2

Cs_, Cp_, Vsum = combine_picture()
assert close(Cs_, 1.2e-6, 1e-9) and close(Cp_, 5e-6, 1e-9) and close(Vsum, 10.0, 1e-9)

In [ ]:
interact(combine_picture, C1_uF=FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description="C₁ (µF)"),
         C2_uF=FloatSlider(value=3.0, min=0.5, max=10.0, step=0.5, description="C₂ (µF)"),
         V=FloatSlider(value=10.0, min=1.0, max=20.0, step=1.0, description="V (volts)"));

## Charging a capacitor a spoonful at a time

Carry the charge across in equal spoonfuls. Each one is carried against the voltage already there, which grows as the
charge builds. The work is a staircase under the straight V–Q line, and it creeps up to the triangle's area, ½CV²,
from below. Slide the number of spoonfuls.

In [ ]:
def spoons_picture(n=10, C=5e-6, V=10.0):
    Q = C * V; dq = Q / n
    qs = np.arange(n) * dq
    work = float(np.sum(qs / C * dq))
    fig, ax = plt.subplots(figsize=(9.5, 4.2))
    fig.subplots_adjust(left=0.09, right=0.98, top=0.84, bottom=0.17)
    ax.bar((qs + dq / 2) * 1e6, qs / C, width=dq * 1e6, color=VIOLET, alpha=0.6, edgecolor=SLATE)
    ax.plot([0, Q * 1e6], [0, V], color=CHALK)
    ax.set_xlabel("charge moved (µC)"); ax.set_ylabel("voltage (V)")
    fig.suptitle(f"{n} spoonfuls: {work * 1e6:.2f} µJ = {work / energy(C, V):.4f} of ½CV² = {energy(C, V) * 1e6:.0f} µJ",
                 x=0.09, ha="left", color=GOLD)
    footnote(fig, "added up spoonful by spoonful"); plt.show()
    return work / energy(C, V)

assert round(spoons_picture(10), 4) == 0.9

In [ ]:
interact(spoons_picture, n=IntSlider(value=10, min=2, max=200, step=1, description="spoonfuls"));

## The camera flash: change the capacitor, watch the energy

The course's opening question. A disposable camera's flash capacitor (160 µF, charged to about 300 V) holds
½CV² of energy. The battery fills it gently over a few seconds; the flash empties it in about a thousandth of a second.
Compare the two rates.

In [ ]:
def flash_picture(C_uF=160.0, V=300.0, t_charge=5.0, t_flash_ms=1.0):
    U = energy(C_uF * 1e-6, V)
    P_in, P_out = U / t_charge, U / (t_flash_ms / 1000)
    fig, ax = plt.subplots(figsize=(9.5, 3.6))
    fig.subplots_adjust(left=0.27, right=0.97, top=0.78, bottom=0.2)
    ax.barh([1, 0], [P_in, P_out], color=[DIM, GOLD]); ax.set_xscale("log")
    ax.set_yticks([1, 0]); ax.set_yticklabels(["filling (battery side)", "the flash"]); ax.set_xlabel("watts")
    fig.suptitle(f"U = {U:.2f} J; in at {P_in:.2f} W, out at {P_out:,.0f} W: {P_out / P_in:,.0f} times faster",
                 x=0.03, ha="left", fontsize=14, color=GOLD)
    footnote(fig, "computed · 160 µF, 300 V: a disposable camera's flash"); plt.show()
    return U, P_in, P_out

U_h, Pin_h, Pout_h = flash_picture()
assert round(U_h, 2) == 7.2 and round(Pout_h) == 7200 and round(Pout_h / Pin_h) == 5000

In [ ]:
interact(flash_picture, C_uF=FloatSlider(value=160.0, min=50.0, max=400.0, step=10.0, description="C (µF)"),
         V=FloatSlider(value=300.0, min=100.0, max=330.0, step=10.0, description="V (volts)"),
         t_charge=FloatSlider(value=5.0, min=1.0, max=15.0, step=0.5, description="fill (s)"),
         t_flash_ms=FloatSlider(value=1.0, min=0.05, max=2.0, step=0.05, description="flash (ms)"));

## Sources
Every exercise and set-up here is our own. The constants are the course's rounded values (k, ε₀, e and the electron's
mass from CODATA 2022, physics.nist.gov/cuu/Constants); air's breakdown field, about 3 × 10⁶ V/m in dry air at sea
level, is a textbook value (Tipler). The flash capacitor's rating (160 µF, 330 V, charged to about 300 V) is a Kodak
disposable camera's, from Sam Goldwasser's Strobe FAQ (repairfaq.org). The simulator is the course's: fourth-order
Runge–Kutta with a fixed step.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib, sys
_src = pathlib.Path("../../../episodes/ph14_potential_capacitance/practice.py")
if _src.exists():
    sys.path.insert(0, str(_src.resolve().parents[2]))
    _spec = importlib.util.spec_from_file_location("ph14_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"], _q["parts"], _q["hint"]) == (
            _e["q"], _e["working"], _e["function"], _e["tier"], _e["parts"], _e["hint"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 * max(1, abs(b)) for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX)
    assert (K, EPS0, E, ME, g, E_SPARK) == (_P.K, _P.EPS0, _P.E, _P.ME, _P.g, _P.E_SPARK)
    assert close(_P.FLASH["U"], U_h, 1e-12) and close(_P.SER["C"], Cs_, 1e-12)
    print("matches the course's practice set")